# Model Development and Evaluation

We train and compare multiple classification models to predict `is_high_risk` using customer behavioral features.


In [ ]:
import os
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split

sys.path.append(str(Path("..").resolve()))

from src.data_processing import run_pipeline

processed_path = Path("..") / "data" / "processed" / "customer_features.csv"
raw_path = Path("..") / "data" / "raw" / "alternate_data.csv"

if not processed_path.exists():
    print("Processed data not found. Generating features from raw data...")
    run_pipeline(str(raw_path), str(processed_path.parent))

df = pd.read_csv(processed_path)
print("Processed dataset shape:", df.shape)
print(df["is_high_risk"].value_counts(normalize=True))
df.head()


## Model Training Workflow

We split the processed customer-level dataset and train three candidate classifiers.


In [ ]:
target = "is_high_risk"
feature_cols = [c for c in df.columns if c not in ["CustomerId", target]]
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train class distribution:", y_train.value_counts(normalize=True).to_dict())

def make_model_grid():
    return {
        "LogisticRegression": (
            LogisticRegression(max_iter=1000, solver="liblinear", random_state=42),
            {"C": [0.01, 0.1, 1.0]},
        ),
        "RandomForest": (
            RandomForestClassifier(random_state=42),
            {"n_estimators": [100, 200], "max_depth": [None, 8]},
        ),
        "GradientBoosting": (
            GradientBoostingClassifier(random_state=42),
            {"n_estimators": [100], "learning_rate": [0.1], "max_depth": [3]},
        ),
    }

def evaluate_model(estimator, X, y):
    y_pred = estimator.predict(X)
    y_prob = None
    if hasattr(estimator, "predict_proba"):
        y_prob = estimator.predict_proba(X)[:, 1]
    elif hasattr(estimator, "decision_function"):
        y_prob = estimator.decision_function(X)

    return {
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob) if y_prob is not None else None,
    }


In [ ]:
model_grid = make_model_grid()
results = []
best_model = None
best_score = -1.0

for name, (estimator, params) in model_grid.items():
    print(f"Training {name}...")
    search = GridSearchCV(
        estimator, params, scoring="roc_auc", cv=3, n_jobs=-1, verbose=0
    )
    search.fit(X_train, y_train)
    clf = search.best_estimator_
    metrics = evaluate_model(clf, X_test, y_test)
    print(name, metrics)
    print(classification_report(y_test, clf.predict(X_test)))
    results.append({
        "model": name,
        "best_params": search.best_params_,
        "metrics": metrics,
    })
    if metrics["roc_auc"] is not None and metrics["roc_auc"] > best_score:
        best_score = metrics["roc_auc"]
        best_model = clf

results_df = pd.DataFrame(results)
results_df


In [ ]:
output_dir = Path("..") / "models"
output_dir.mkdir(parents=True, exist_ok=True)
model_file = output_dir / "best_model.joblib"
feature_file = output_dir / "feature_columns.csv"

joblib.dump(best_model, model_file)
pd.Series(feature_cols).to_csv(feature_file, index=False)
print("Saved best model to", model_file)
print("Saved feature columns to", feature_file)


## Sample Inference

Use the trained model to score a new customer record from the processed feature set.


In [ ]:
sample_index = 0
sample = X_test.iloc[[sample_index]]
probability = best_model.predict_proba(sample)[:, 1][0] if hasattr(best_model, "predict_proba") else best_model.decision_function(sample)[0]
label = int(probability >= 0.5)
print("Sample customer id:", df.loc[sample.index[0], "CustomerId"])
print("Predicted is_high_risk:", label)
print("Risk probability:", probability)
